# Commercial MT reference baseline — zero-shot


---
## 1 — Setup

API-only

In [1]:
!git pull

Already up to date.


In [2]:
# Minimal install: no torch, no transformers, no GPU. Takes seconds, not minutes.
!pip install -q anthropic==0.109.1 openai==2.41.1 google-genai sacrebleu==2.6.0 \
    PyYAML==6.0.3 python-dotenv==1.2.2 numpy


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import getpass, logging, os
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')
logging.getLogger('httpx').setLevel(logging.WARNING)   # one INFO line per call otherwise

In [1]:
%cd /home/prnamhr/projects/Style-Aware-MT

/home/prnamhr/projects/Style-Aware-MT


### Pre-flight

In [2]:
import yaml, pathlib, json
from src.infer.run import resolve_out_name

CONFIG = 'configs/commercial_haiku_zeroshot.yaml'
cfg = yaml.safe_load(pathlib.Path(CONFIG).read_text())
gen = cfg['generator']
name = resolve_out_name('zeroshot', cfg)
out_path = pathlib.Path(cfg['output']['dir']) / f"{name}_val.jsonl"

assert gen['temperature'] == 0.0, 'temperature must be 0.0 -- see the config comment'
assert cfg['data']['eval_file'].endswith('val.jsonl'), 'test split stays sealed'
assert cfg['data']['limit'] is None, 'limit must be null for the full 1,323 segments'
assert name != 'zeroshot', 'output name would collide with the Qwen zero-shot file'
assert cfg['prompt']['style_instruction_file'] == \
    yaml.safe_load(pathlib.Path('configs/base_qwen.yaml').read_text())['prompt']['style_instruction_file'], \
    'style instruction differs from the Qwen run -- the comparison would not be model-only'

n_eval = sum(1 for _ in open(cfg['data']['eval_file']))
print(f"model      : {gen['model']} (temperature {gen['temperature']})")
print(f"condition  : zeroshot on {n_eval} val segments")
print(f"writes     : {out_path}")
print(f"existing   : outputs/zeroshot_val.jsonl untouched")

/home/prnamhr/projects/Style-Aware-MT/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model      : claude-haiku-4-5 (temperature 0.0)
condition  : zeroshot on 1323 val segments
writes     : outputs/commercial_haiku_val.jsonl
existing   : outputs/zeroshot_val.jsonl untouched


In [3]:
# Cost estimate from the sweep harness's own token model, at the real segment count.
from src.infer.sweep import estimate_cost
print(f"estimated generation cost: ${estimate_cost('claude-haiku-4-5', 0, n_eval):.2f}  ({n_eval} calls)")

estimated generation cost: $0.58  (1323 calls)


---
## 2 — Generate

In [4]:
!python3 manage.py infer --condition zeroshot --config configs/commercial_haiku_zeroshot.yaml

Output name overridden: condition 'zeroshot' -> outputs/commercial_haiku_val.jsonl
Generating 1323 translations with claude-haiku-4-5 (zeroshot) ...
  5/1323
  10/1323
  15/1323
  20/1323
  25/1323
  30/1323
  35/1323
  40/1323
  45/1323
  50/1323
  55/1323
  60/1323
  65/1323
  70/1323
  75/1323
  80/1323
  85/1323
  90/1323
  95/1323
  100/1323
  105/1323
  110/1323
  115/1323
  120/1323
  125/1323
  130/1323
  135/1323
  140/1323
  145/1323
  150/1323
  155/1323
  160/1323
  165/1323
  170/1323
  175/1323
  180/1323
  185/1323
  190/1323
  195/1323
  200/1323
  205/1323
  210/1323
  215/1323
  220/1323
  225/1323
  230/1323
  235/1323
  240/1323
  245/1323
  250/1323
  255/1323
  260/1323
  265/1323
  270/1323
  275/1323
  280/1323
  285/1323
  290/1323
  295/1323
  300/1323
  305/1323
  310/1323
  315/1323
  320/1323
  325/1323
  330/1323
  335/1323
  340/1323
  345/1323
  350/1323
  355/1323
  360/1323
  365/1323
  370/1323
  375/1323
  380/1323
  385/1323
  390/1323
  395/1323
  

In [5]:
# Actual spend and failure count for this session.
import json
u = json.load(open('outputs/commercial_haiku_val_usage.json'))
print(u)
rows = [json.loads(l) for l in open('outputs/commercial_haiku_val.jsonl')]
failed = [i for i, r in enumerate(rows) if r.get('error') or not r['prediction'].strip()]
print(f"rows: {len(rows)}  |  empty/failed: {len(failed)}")
if failed:
    print('failed indices:', failed[:20])
    print('delete those lines and re-run the cell above to retry them')

{'condition': 'zeroshot', 'output_name': 'commercial_haiku', 'model': 'claude-haiku-4-5', 'calls': 1323, 'prompt_tokens': 375530, 'completion_tokens': 51220, 'cost_usd': 0.6316}
rows: 1323  |  empty/failed: 0


---
## 3 — Verify the comparison is model-only

In [6]:
import json
qwen = [json.loads(l) for l in open('outputs/zeroshot_val.jsonl')]
comm = [json.loads(l) for l in open('outputs/commercial_haiku_val.jsonl')]

assert len(qwen) == len(comm), f'length mismatch: {len(qwen)} vs {len(comm)}'
assert all(a['input'] == b['input'] for a, b in zip(qwen, comm)), 'sources diverge'
assert all(a['output'] == b['output'] for a, b in zip(qwen, comm)), 'references diverge'
assert {r['condition'] for r in comm} == {'zeroshot'}, 'condition label is not zeroshot'

print(f"aligned on {len(qwen)} segments, identical sources and references")
print(f"  qwen model      : {sorted({r['model'] for r in qwen})}")
print(f"  commercial model: {sorted({r['model'] for r in comm})}")
print('\nsegment 0')
print('  reference :', qwen[0]['output'][:160])
print('  qwen      :', qwen[0]['prediction'][:160])
print('  haiku     :', comm[0]['prediction'][:160])

aligned on 1323 segments, identical sources and references
  qwen model      : ['Qwen/Qwen2.5-7B-Instruct']
  commercial model: ['claude-haiku-4-5']

segment 0
  reference : The essence of the divine mysteries in the journeys of ascent set forth for those who long to draw nigh unto God, the Almighty, the Ever-Forgiving—blessed be th
  qwen      : Blessed are the righteous who drink from these rivers, O Thou, the Almighty, the Forgiver, unto whom approacheth none save those who draw nigh by power and migh
  haiku     : The Gems of Mysteries in the Stages of Ascent for whosoever desireth to draw nigh unto God, the All-Powerful, the All-Forgiving—a felicitation unto the righteou


---
## 4 — Free metrics: chrF, BLEU, stylometrics

Local and free. `--conditions` takes the file stem, so the overridden output name
is the condition name from here on.

In [7]:
CONDS = 'zeroshot commercial_haiku'
!python3 manage.py eval         --conditions {CONDS} --split val
!python3 manage.py stylometrics --conditions {CONDS} --split val

condition         n     BLEU   chrF   marker_rate  ref_marker_rate
------------------------------------------------------------------
zeroshot          1323  10.27  36.42  1.4          0.79           
commercial_haiku  1323  18.06  45.24  1.11         0.79           
label             n     lex_density  lex_density_sd  ttr     ttr_sd  root_ttr  root_ttr_sd  sent_len_mean  sent_len_mean_sd  sent_len_var  sent_len_var_sd  marker_rate  marker_rate_sd  stylo_dist
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
zeroshot          1323  0.4025       0.0999          0.8434  0.1222  3.858     0.9711       23.7585        19.5748           4.5813        32.7304          0.0637       0.0817          0.6518    
commercial_haiku  1323  0.3928       0.0984          0.8241  0.125   3.9102    0.9765       25.7984        17.3883           3.6542        35.55

---
## 5 — Judge Φ

In [8]:
!python3 manage.py judge --conditions commercial_haiku --split val --config configs/judge_eval.yaml

Judging 1323 segments for commercial_haiku with claude-haiku-4-5 ...
  commercial_haiku Φ 3.333  (coverage 100%)
Judge usage: {'calls': 1323, 'prompt_tokens': 628348, 'completion_tokens': 143576, 'cost_usd': 1.3462}
Wrote results/judge_val.json


---
## 6 — COMET + paired bootstrap *(separate environment)*

In [9]:
!pip install -q -r requirements-comet.txt
# Runtime -> Restart session, then re-run the %cd cell before continuing.


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [2]:
CONDS = 'zeroshot commercial_haiku'
!python3 manage.py comet --conditions {CONDS} --split val

/home/prnamhr/projects/Style-Aware-MT
Fetching 5 files: 100%|███████████████████████| 5/5 [00:00<00:00, 91578.69it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/prnamhr/projects/Style-Aware-MT/.venv/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platf

In [ ]:
# Adequacy proxies (chrF/BLEU) and register stylometrics — free/local, no COMET.
CONDS = 'zeroshot random_fewshot knn_fewshot afsp_margin afsp_full '
!python manage.py eval         --conditions {CONDS} --split val
!python manage.py stylometrics --conditions {CONDS} --split val

condition       n     BLEU   chrF   marker_rate  ref_marker_rate
----------------------------------------------------------------
zeroshot        1323  10.27  36.42  1.4          0.79           
random_fewshot  1323  11.64  37.52  1.14         0.79           
knn_fewshot     1323  13.99  39.82  0.9          0.79           
afsp_margin     1323  13.69  39.68  0.96         0.79           
afsp_full       1323  14.52  39.99  0.98         0.79           
label           n     lex_density  lex_density_sd  ttr     ttr_sd  root_ttr  root_ttr_sd  sent_len_mean  sent_len_mean_sd  sent_len_var  sent_len_var_sd  marker_rate  marker_rate_sd  stylo_dist
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
zeroshot        1323  0.4025       0.0999          0.8434  0.1222  3.858     0.9711       23.7585        19.5748           4.5813        32.7304          0.0